# Hyperparameter Tuning: Customer Churn

> **Authorship note.** This notebook was written by Claude (an AI assistant) on 22 November 2026 at the repository owner's request, as a reference implementation. It replaces a notebook that closely followed a public tutorial. It is not the owner's own work, and the owner has not yet reworked it.

Goal: tune three classifiers on the owner's customer-churn practice dataset with grid search and randomized search, compare them with cross-validation, and check the winners once on a held-out test set. The data file is a copy of `01_Supervised_Learning/Logistic_Regression/04_logistic_customer_churn.csv`.

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import randint, loguniform

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

## 1. Load the data

In [2]:
df = pd.read_csv("04_logistic_customer_churn.csv")
X = df.drop(columns=["Churn"])
y = df["Churn"]

print("Rows, columns:", df.shape)
print(y.value_counts(normalize=True).round(3).rename("share"))
df.head()

Rows, columns: (250, 6)
Churn
0    0.744
1    0.256
Name: share, dtype: float64


,Monthly_Bill,Tenure_Months,Support_Calls,Late_Payments,Monthly_Usage_Hours,Churn
0,27.92,34,2,1,63.53,0
1,153.33,46,3,2,55.53,0
2,155.03,4,6,3,62.56,1
3,65.59,30,5,0,28.06,0
4,94.06,72,2,2,41.90,0


## 2. Hold out a test set

Only about a quarter of the customers churn, so `stratify=y` keeps that ratio in both parts. The search below uses the training part only.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("Train:", X_train.shape, " Test:", X_test.shape)

Train: (200, 5)  Test: (50, 5)


## 3. Choose the score

With imbalanced classes, accuracy can look good even if most churners are missed. ROC AUC measures how well the model ranks churners above non-churners, so it is used to compare settings.

In [4]:
baseline = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
base_scores = cross_val_score(baseline, X_train, y_train, cv=cv, scoring="roc_auc")
print(f"Baseline logistic regression - mean CV ROC AUC: {base_scores.mean():.3f} (std {base_scores.std():.3f})")

Baseline logistic regression - mean CV ROC AUC: 0.861 (std 0.047)


## 4. Grid search: logistic regression

A grid search tries every combination listed. `C` is the inverse regularisation strength; `class_weight='balanced'` gives the rarer churn class more weight.

In [5]:
lr_search = GridSearchCV(
    make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    param_grid={
        "logisticregression__C": [0.01, 0.1, 1, 10, 100],
        "logisticregression__class_weight": [None, "balanced"],
    },
    cv=cv, scoring="roc_auc")
lr_search.fit(X_train, y_train)
print("Best parameters:", lr_search.best_params_)
print(f"Best mean CV ROC AUC: {lr_search.best_score_:.3f}")

cols = ["param_logisticregression__C", "param_logisticregression__class_weight", "mean_test_score", "rank_test_score"]
pd.DataFrame(lr_search.cv_results_)[cols].sort_values("rank_test_score").head(5).round(3)

Best parameters: {'logisticregression__C': 1, 'logisticregression__class_weight': None}
Best mean CV ROC AUC: 0.861


,param_logisticregression__C,param_logisticregression__class_weight,mean_test_score,rank_test_score
4,1.0,None,0.861,1
6,10.0,None,0.860,2
8,100.0,None,0.860,2
7,10.0,balanced,0.860,4
9,100.0,balanced,0.860,4


## 5. Randomized search: random forest

A random forest has more settings than a grid can cover cheaply. `RandomizedSearchCV` samples 25 random combinations from the ranges below.

In [6]:
rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions={
        "n_estimators": randint(100, 500),
        "max_depth": [None, 3, 5, 8],
        "min_samples_leaf": randint(1, 10),
        "max_features": ["sqrt", "log2", None],
    },
    n_iter=25, cv=cv, scoring="roc_auc", random_state=42, n_jobs=-1)
rf_search.fit(X_train, y_train)
print("Best parameters:", rf_search.best_params_)
print(f"Best mean CV ROC AUC: {rf_search.best_score_:.3f}")

Best parameters: {'max_depth': 3, 'max_features': 'sqrt', 'min_samples_leaf': 9, 'n_estimators': 148}
Best mean CV ROC AUC: 0.810


## 6. Randomized search: SVM

`C` and `gamma` are sampled on a log scale, because useful values span several orders of magnitude.

In [7]:
svm_search = RandomizedSearchCV(
    make_pipeline(StandardScaler(), SVC()),
    param_distributions={
        "svc__C": loguniform(1e-2, 1e2),
        "svc__gamma": loguniform(1e-3, 1e0),
        "svc__class_weight": [None, "balanced"],
    },
    n_iter=25, cv=cv, scoring="roc_auc", random_state=42)
svm_search.fit(X_train, y_train)
print("Best parameters:", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in svm_search.best_params_.items()})
print(f"Best mean CV ROC AUC: {svm_search.best_score_:.3f}")

Best parameters: {'svc__C': np.float64(13.1451), 'svc__class_weight': None, 'svc__gamma': np.float64(0.0029)}
Best mean CV ROC AUC: 0.858


## 7. Compare the tuned models on the test set

The test set is used once, after all tuning is finished. ROC AUC uses the models' scores; F1 and accuracy use their 0/1 predictions.

In [8]:
models = {"Logistic regression (grid)": lr_search, "Random forest (randomized)": rf_search, "SVM (randomized)": svm_search}
rows = []
for name, search in models.items():
    est = search.best_estimator_
    scores = est.predict_proba(X_test)[:, 1] if hasattr(est, "predict_proba") else est.decision_function(X_test)
    y_pred = est.predict(X_test)
    rows.append({"model": name, "best CV ROC AUC": search.best_score_,
                 "test ROC AUC": roc_auc_score(y_test, scores),
                 "test F1 (churn)": f1_score(y_test, y_pred),
                 "test accuracy": accuracy_score(y_test, y_pred)})
pd.DataFrame(rows).round(3)

,model,best CV ROC AUC,test ROC AUC,test F1 (churn),test accuracy
0,Logistic regression (grid),0.861,0.634,0.000,0.68
1,Random forest (randomized),0.810,0.628,0.000,0.74
2,SVM (randomized),0.858,0.663,0.143,0.76


## Summary

- Grid search is thorough for a few settings; randomized search covers larger spaces with a fixed budget.
- Keep scaling inside the pipeline so cross-validation stays honest.
- Pick the score that matches the goal (here ROC AUC for an imbalanced target), and use the test set only once at the end.
- With only 250 customers, differences of a few hundredths between models are within the noise of cross-validation.

All numbers above come from running this notebook.